# 0. Создание исходных данных и загрузка в S3

Создаём 6 CSV-файлов по 1000 событий с помощью генератора.

Загружаем созданные файлы в S3-хранилище RustFS с помощью  upload_to_s3.py.

Проверяем, что все 6 файлов успешно загружены.

Исходные данные сохраняются по адресу s3://dwh-homework/raw/user_events/.


In [1]:
import subprocess
import sys

print("Создание 6000 событий")

subprocess.run([
    sys.executable,
    "generate_event_log_1.py",
    "--rows", "6000",
    "--output-dir", "./event_log",
    "--seed", "42",
    "--overwrite"
], check=True)

print("\nЗагрузка CSV-файлов в RustFS")

subprocess.run([
    sys.executable,
    "upload_to_s3.py"
], check=True)

print("\nПодготовка данных завершена.")

Создание 6000 событий

Загрузка CSV-файлов в RustFS

Подготовка данных завершена.


## 0.1 Импорт библиотек и настройка окружения

In [2]:
# делаем pyspark доступным для jupyter
import os
import sys
from pathlib import Path

spark_home = Path(os.environ.get("SPARK_HOME", "/usr/local/spark-3.5.0-bin-hadoop3"))
if not spark_home.exists():
    candidates = sorted(Path("/usr/local").glob("spark*"))
    if not candidates:
        raise FileNotFoundError("Не найден Spark в /usr/local")
    spark_home = candidates[0]
os.environ["SPARK_HOME"] = str(spark_home)

lib_dir = spark_home / "python" / "lib"
pyspark_zip = lib_dir / "pyspark.zip"
py4j_zips = sorted(lib_dir.glob("py4j-*.zip"))
if not pyspark_zip.is_file() or not py4j_zips:
    raise FileNotFoundError(f"Не найдены pyspark.zip / py4j-*.zip в {lib_dir}")

spark_paths = [str(pyspark_zip), *(str(p) for p in py4j_zips)]
for item in reversed(spark_paths):
    if item not in sys.path:
        sys.path.insert(0, item)
# путь для дочерним процессам Spark
os.environ["PYTHONPATH"] = os.pathsep.join(
    spark_paths + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else [])
)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    "--packages org.apache.hadoop:hadoop-aws:3.3.4 pyspark-shell"
)

import pyspark
print("PySpark импортирован:", pyspark.__version__)
print("Библиотеки Spark найдены.")

PySpark импортирован: 3.5.0
Библиотеки Spark найдены.


## 1. Создаём SparkSession и подключаем S3

In [3]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

BUCKET = "dwh-homework"
SOURCE = f"s3a://{BUCKET}/raw/user_events/"
TARGET = f"s3a://{BUCKET}/scd2/users/"

spark = (
    SparkSession.builder
    .appName("DWH-HW2.1-Notebook")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
config = {
    "fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "fs.s3a.endpoint": os.environ.get("S3_ENDPOINT_URL", "http://rustfs:9000"),
    "fs.s3a.access.key": os.environ["AWS_ACCESS_KEY_ID"],
    "fs.s3a.secret.key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "fs.s3a.path.style.access": "true",
    "fs.s3a.connection.ssl.enabled": "false",
    "fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
}
for name, value in config.items():
    hadoop_conf.set(name, value)

print("spark:", spark.version)
print("источник:", SOURCE)
print("результат:", TARGET)

spark: 3.5.0
источник: s3a://dwh-homework/raw/user_events/
результат: s3a://dwh-homework/scd2/users/


## 2. Проверяем исходные CSV
В S3 бакете dwh-homework исходные CSV лежат как raw/user_events/. Файлы были сгенерированы с seed=42 и загружены через upload_to_s3.py


In [4]:
schema = StructType([
    StructField("event_time", StringType(), False),
    StructField("user_id", LongType(), False),
    StructField("user_fio", StringType(), True),
    StructField("user_email", StringType(), True),
    StructField("user_login", StringType(), True),
])

raw = spark.read.option("header", "true").schema(schema).csv(SOURCE)
events = raw.withColumn(
    "event_time",
    F.to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ssX")
)
if events.filter(F.col("event_time").isNull() | F.col("user_id").isNull()).limit(1).count():
    raise ValueError("Есть некорректные event_time или user_id")

print("событий:", events.count())
print("различных пользователей:", events.select("user_id").distinct().count())
print("поля:", ", ".join(events.columns))

событий: 6000
различных пользователей: 300
поля: event_time, user_id, user_fio, user_email, user_login


### 2.1. Сортируем события по времени



In [5]:
events.orderBy("event_time").show(8, truncate=False)

+-------------------+-------+---------------+----------------------------------+-----------+
|event_time         |user_id|user_fio       |user_email                        |user_login |
+-------------------+-------+---------------+----------------------------------+-----------+
|2026-09-01 00:00:03|58     |Aiden Howard   |aiden.howard.58@mail.example      |ahoward58  |
|2026-09-01 00:00:04|72     |Ashley Williams|ashley.williams.72@webmail.example|awilliams72|
|2026-09-01 00:00:05|217    |Alexander Baker|alexander.baker.217@example.com   |abaker217  |
|2026-09-01 00:00:06|120    |Lucas Stewart  |lucas.stewart.120@example.com     |lstewart120|
|2026-09-01 00:00:09|102    |Taylor Wright  |taylor.wright.102@webmail.example |twright102 |
|2026-09-01 00:00:11|113    |Justin Scott   |justin.scott.113@inbox.example    |jscott113  |
|2026-09-01 00:00:12|82     |Joseph King    |joseph.king.82@inbox.example      |jking82    |
|2026-09-01 00:00:13|111    |Henry Clark    |henry.clark.111@example.c

## 3. Проверяем события по пользователю

Возьмём user_id=5. У него 23 много события, но смена состояния происходит один раз - 

email меняется с cameron.butler.5@post.example на cameron.butler.5.r1@example.com

SCD2 нужно сохранить только две версии.

In [6]:
u5 = events.filter(F.col("user_id") == 5).orderBy("event_time")
print("Событий у user_id=5:", u5.count())
u5.select("event_time", "user_email", "user_login").show(8, truncate=False)

Событий у user_id=5: 23
+-------------------+-------------------------------+----------+
|event_time         |user_email                     |user_login|
+-------------------+-------------------------------+----------+
|2026-09-01 00:24:13|cameron.butler.5@post.example  |cbutler5  |
|2026-09-01 00:27:52|cameron.butler.5@post.example  |cbutler5  |
|2026-09-01 00:29:38|cameron.butler.5.r1@example.com|cbutler5  |
|2026-09-01 00:29:48|cameron.butler.5.r1@example.com|cbutler5  |
|2026-09-01 00:31:43|cameron.butler.5.r1@example.com|cbutler5  |
|2026-09-01 00:37:28|cameron.butler.5.r1@example.com|cbutler5  |
|2026-09-01 00:39:44|cameron.butler.5.r1@example.com|cbutler5  |
|2026-09-01 00:40:11|cameron.butler.5.r1@example.com|cbutler5  |
+-------------------+-------------------------------+----------+
only showing top 8 rows



## 4. Поиск изменений с Window + lag()

In [7]:
w = Window.partitionBy("user_id").orderBy("event_time")
state = F.struct("user_fio", "user_email", "user_login")

with_prev = (
    events
    .withColumn("_state", state)
    .withColumn("_prev_state", F.lag("_state").over(w))
    .withColumn("_prev_email", F.lag("user_email").over(w))
    .withColumn(
        "is_new_version",
        F.col("_prev_state").isNull()
        | (~F.col("_state").eqNullSafe(F.col("_prev_state")))
    )
)

(with_prev.filter(F.col("user_id") == 5)
 .orderBy("event_time")
 .select("event_time", "user_email", "_prev_email", "is_new_version")
 .show(8, truncate=False))

+-------------------+-------------------------------+-------------------------------+--------------+
|event_time         |user_email                     |_prev_email                    |is_new_version|
+-------------------+-------------------------------+-------------------------------+--------------+
|2026-09-01 00:24:13|cameron.butler.5@post.example  |NULL                           |true          |
|2026-09-01 00:27:52|cameron.butler.5@post.example  |cameron.butler.5@post.example  |false         |
|2026-09-01 00:29:38|cameron.butler.5.r1@example.com|cameron.butler.5@post.example  |true          |
|2026-09-01 00:29:48|cameron.butler.5.r1@example.com|cameron.butler.5.r1@example.com|false         |
|2026-09-01 00:31:43|cameron.butler.5.r1@example.com|cameron.butler.5.r1@example.com|false         |
|2026-09-01 00:37:28|cameron.butler.5.r1@example.com|cameron.butler.5.r1@example.com|false         |
|2026-09-01 00:39:44|cameron.butler.5.r1@example.com|cameron.butler.5.r1@example.com|false 

## 5. Оставляем только начало каждой версии

С помощью is_new_version == true удаляем повторные события, в которых профиль не менялся.


In [8]:
changed = with_prev.filter(F.col("is_new_version")).select(
    "event_time", "user_id", "user_fio", "user_email", "user_login"
)
print("Начал версий:", changed.count())
changed.filter("user_id = 5").orderBy("event_time").show(truncate=False)

Начал версий: 350
+-------------------+-------+--------------+-------------------------------+----------+
|event_time         |user_id|user_fio      |user_email                     |user_login|
+-------------------+-------+--------------+-------------------------------+----------+
|2026-09-01 00:24:13|5      |Cameron Butler|cameron.butler.5@post.example  |cbutler5  |
|2026-09-01 00:29:38|5      |Cameron Butler|cameron.butler.5.r1@example.com|cbutler5  |
+-------------------+-------+--------------+-------------------------------+----------+



## 6. Строим интервалы SCD2 с помощью lead()



In [9]:
versions = (
    changed
    .withColumn("valid_from", F.col("event_time"))
    .withColumn("valid_to", F.lead("event_time").over(w))
    .withColumn("is_current", F.col("valid_to").isNull())
    .select(
        "user_id", "user_fio", "user_email", "user_login",
        "valid_from", "valid_to", "is_current"
    )
)
print("Версий SCD2:", versions.count())
versions.filter("user_id = 5").orderBy("valid_from").show(truncate=False)

Версий SCD2: 350
+-------+--------------+-------------------------------+----------+-------------------+-------------------+----------+
|user_id|user_fio      |user_email                     |user_login|valid_from         |valid_to           |is_current|
+-------+--------------+-------------------------------+----------+-------------------+-------------------+----------+
|5      |Cameron Butler|cameron.butler.5@post.example  |cbutler5  |2026-09-01 00:24:13|2026-09-01 00:29:38|false     |
|5      |Cameron Butler|cameron.butler.5.r1@example.com|cbutler5  |2026-09-01 00:29:38|NULL               |true      |
+-------+--------------+-------------------------------+----------+-------------------+-------------------+----------+



## 7. Сохраняем SCD2 в RustFS


In [10]:
WRITE_OUTPUT = True  # False если нужен прошлый результат из S3

if WRITE_OUTPUT:
    versions.write.mode("overwrite").parquet(TARGET)
    print("Parquet перезаписан:", TARGET)
else:
    print("Запись пропущена — читаем ранее сохранённую таблицу.")

result = spark.read.parquet(TARGET)
print("Строк после чтения из S3:", result.count())
result.createOrReplaceTempView("users_scd2")

Parquet перезаписан: s3a://dwh-homework/scd2/users/
Строк после чтения из S3: 350


## 8. Подсчет строк в итоговой таблице с SQL


In [11]:
spark.sql("SELECT COUNT(*) AS scd2_rows FROM users_scd2").show()

+---------+
|scd2_rows|
+---------+
|      350|
+---------+



### 8.1. Подсчёт количества пользователей и записей

Проверим состав полученной таблицы SCD2 с историей изменений пользователей.
В результате получаем 300 пользователей, 300 актуальных записей и 50 устаревших(исторических). 

In [12]:
spark.sql("""
SELECT
    COUNT(DISTINCT user_id) AS users,
    SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows,
    SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS historical_rows
FROM users_scd2
""").show()

+-----+------------+---------------+
|users|current_rows|historical_rows|
+-----+------------+---------------+
|  300|         300|             50|
+-----+------------+---------------+



## 9. Проверка правильности истории изменений пользователей

Для этого должны выполнятся четыре условия.
1. Одна актуальная запись на пользователя.
2. Если is_current = true, поле valid_to должно быть пустым. Для всех предыдущих версий должна быть указана дата окончания действия.
3. Дата окончания предыдущей версии valid_to должна быть позже даты её начала valid_from.
4. Дата окончания одной версии должна совпадать с датой начала следующей версии того же пользователя.
В следующей ячейке подсчитываем количество нарушений каждого условия. Если все значения равны нулю, перечисленные проверки пройдены.

In [13]:
current_per_user = (
    result.groupBy("user_id")
    .agg(F.sum(F.col("is_current").cast("int")).alias("n_current"))
)
bad_current = current_per_user.filter("n_current <> 1").count()
bad_current_flags = result.filter(
    ~F.col("is_current").eqNullSafe(F.col("valid_to").isNull())
).count()
bad_intervals = result.filter(
    F.col("valid_to").isNotNull() & (F.col("valid_to") <= F.col("valid_from"))
).count()

w_version = Window.partitionBy("user_id").orderBy("valid_from")
with_next = result.withColumn("next_from", F.lead("valid_from").over(w_version))
bad_boundaries = with_next.filter(
    F.col("next_from").isNotNull()
    & (~F.col("valid_to").eqNullSafe(F.col("next_from")))
).count()

print("Пользователи с не 1 актуальной версией:", bad_current)
print("Некорректные is_current/valid_to:", bad_current_flags)
print("Неположительные интервалы:", bad_intervals)
print("Разрывы / пересечения границ:", bad_boundaries)
assert (bad_current, bad_current_flags, bad_intervals, bad_boundaries) == (0, 0, 0, 0)
print("ПРОВЕРКИ ПРОЙДЕНЫ")

Пользователи с не 1 актуальной версией: 0
Некорректные is_current/valid_to: 0
Неположительные интервалы: 0
Разрывы / пересечения границ: 0
ПРОВЕРКИ ПРОЙДЕНЫ


## 10. Поиск пользователя на заданную дату с SQL

Чтобы найти нужную запись, проверяем два условия:
1. valid_from <= :y - момент времени наступил не раньше начала действия записи.
2. valid_to IS NULL OR valid_to > :y - запись ещё действительна


Значения :x и :y передаются в SQL как параметры:
 1. :x - идентификатор пользователя.
 2. :y - дата и время.

В тестовом наборе для X=47, Y=2026-09-01 03:00:00 UTС ожидаем acollins47r1.

In [14]:
from datetime import datetime, timezone

X = 47
Y = datetime(2026, 9, 1, 3, 0, 0, tzinfo=timezone.utc)

SQL_LOGIN_AT_TIME = """
SELECT user_id, user_login, valid_from, valid_to
FROM users_scd2
WHERE user_id = :x
  AND valid_from <= :y
  AND (valid_to IS NULL OR :y < valid_to)
"""

spark.sql(SQL_LOGIN_AT_TIME, args={"x": X, "y": Y}).show(truncate=False)

+-------+------------+-------------------+--------+
|user_id|user_login  |valid_from         |valid_to|
+-------+------------+-------------------+--------+
|47     |acollins47r1|2026-09-01 02:43:54|NULL    |
+-------+------------+-------------------+--------+



### 10.1. Проверка граничных случаев

1. Запрос до первого события пользователя должен вернуть 0 строк.
2. Ровно в момент valid_to старой версии нужно получить новую версию.

В тестовом наборе user_id=5 в 00:29:38 поменял email.

In [15]:
BEFORE = datetime(2026, 8, 31, 23, 0, 0, tzinfo=timezone.utc)
empty_count = spark.sql(SQL_LOGIN_AT_TIME, args={"x": 47, "y": BEFORE}).count()
print("Строк до первого события user_id=47:", empty_count)
assert empty_count == 0

AT_CHANGE = datetime(2026, 9, 1, 0, 29, 38, tzinfo=timezone.utc)
spark.sql("""
SELECT user_id, user_email, valid_from, valid_to
FROM users_scd2
WHERE user_id = :x
  AND valid_from <= :y
  AND (valid_to IS NULL OR :y < valid_to)
""", args={"x": 5, "y": AT_CHANGE}).show(truncate=False)

Строк до первого события user_id=47: 0
+-------+-------------------------------+-------------------+--------+
|user_id|user_email                     |valid_from         |valid_to|
+-------+-------------------------------+-------------------+--------+
|5      |cameron.butler.5.r1@example.com|2026-09-01 00:29:38|NULL    |
+-------+-------------------------------+-------------------+--------+



## 11. Проверка правильности сохранения таблицы SCD2 в S3

После построения таблицы SCD2 необходимо проверить, что все рассчитанные записи были правильно сохранены в S3.

Для этого сравниваем таблицу versions, полученную в результате вычислений Spark с таблицей result, загруженной обратно из S3.

С помощью exceptAll проверяем, нет ли отсутствующих или лишних записей. 

In [16]:
missing_in_s3 = versions.exceptAll(result).count()
extra_in_s3 = result.exceptAll(versions).count()
print("Версий в расчёте, отсутствующих в S3:", missing_in_s3)
print("Лишних версий в S3:", extra_in_s3)
assert missing_in_s3 == extra_in_s3 == 0
print("Совпадение данных подтверждено.")

Версий в расчёте, отсутствующих в S3: 0
Лишних версий в S3: 0
Совпадение данных подтверждено.
